# Lab 01: Call a deployed model in Azure AI Foundry

In this notebook, you send prompts to a deployed model. You use the OpenAI Python SDK and the REST API.

**Duration:** about 20 minutes.

Read [Introduction](https://github.com/Kiran-255666/AgenticAIEngineer/blob/main/Instructions/Exercises/Day01/lab01-azure-ai-foundry.md) before you start. It explains tokens, system messages, and the controls that change model output.

## Before you begin

1. Open this notebook in Visual Studio Code.
2. Install the **Python (Microsoft)** extension and the **Jupyter (Microsoft)** extension.
3. Enable both extensions.
4. Wait for both extensions to finish installing.

> **Note:** Follow these steps if the notebook does not run, no kernel appears, or Jupyter shows an activation error.
>
> 1. Open the Jupyter extension.
> 2. Click the ▼ next to **Uninstall**.
> 3. Select **Install Specific Version...**
> 4. Install Jupyter **v2025.4.0**.
> 5. Reload VS Code.
> 6. Open the notebook again.

### Configure the kernel

1. Click **Select Kernel**.
2. Under **Kernel Source**, select **Python Environments...**
3. Choose **Python 3.12.1**.

You can now run the notebook. Select a cell and press **Shift + Enter** to run it. Run the cells in order.

## Get Your API Key

1. Return to the [Azure AI Foundry portal](https://ai.azure.com/).
2. On the **Home** page, under **Recent works**, select the deployed **gpt-5.4-mini** model.
3. Select the **Details** tab next to **Playground**.
4. Copy the **Key** value.
5. Save the key in Notepad.
6. Return to the notebook and replace the `API_KEY` placeholder with your copied key.

**Note:** The endpoint and deployment name are already preconfigured in the notebook. You only need to provide your API key. Do not modify the other values.

## Calling the model

This notebook does these tasks:

- Install the OpenAI Python SDK.
- Configure the endpoint, key, and deployment name.
- Send a chat request, first as a complete response and then as a stream.
- Ask for a reply in JSON format.
- Run a small parameter sweep.
- Send a request with REST.

## Install packages

Run this cell one time. It installs the `openai` package and the `requests` package.

In [ ]:
# Install the OpenAI SDK (talks to the model) and requests (sends raw HTTP calls).
%pip install --upgrade openai requests

## Configure your endpoint

The endpoint and deployment name are already configured for you.

TODO: Replace `API_KEY` with your actual API key from the Foundry portal.

In [ ]:
# The endpoint and deployment name are preconfigured for you.
ENDPOINT = "https://hakunamatata1234.services.ai.azure.com/openai/v1/"

# TODO: Replace with your actual API key from Foundry > your resource > Keys and Endpoint.
API_KEY = "<YOUR-API-KEY>"

# Preconfigured deployment name.
DEPLOYMENT = "gpt-5.4-mini"

# Normalize the endpoint to /openai/v1, removing any trailing slash or extra path.
ENDPOINT = ENDPOINT.split("/openai/v1")[0] + "/openai/v1"

# Stop if the endpoint is not a valid HTTPS address.
assert ENDPOINT.startswith("https://"), "Please set a valid ENDPOINT"

# Stop if the API key is empty or still a placeholder.
assert API_KEY and API_KEY != "<YOUR-API-KEY>", "Please set your API_KEY"

# Stop if the deployment name is empty.
assert DEPLOYMENT, "Please set your DEPLOYMENT name"

## Initialize a client

The client sends every request to your endpoint with your key.

In [ ]:
# Import the client class from the OpenAI SDK.
from openai import OpenAI

# Create a client that uses your endpoint and your key for every request.
client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

## Send a basic request

The system message sets the role of the model. The user message is the question.

In [ ]:
# Send one request to the Responses API and keep the reply.
response = client.responses.create(
    # Choose the deployed model that answers.
    model=DEPLOYMENT,
    # Send a list of messages, in order.
    input=[
        # The system message sets the role and tone of the model.
        {"role": "system", "content": "You are a concise Azure assistant."},
        # The user message is the question the model answers.
        {"role": "user",   "content": "In one sentence, define a foundation model."}
    ],
    # Limit the length of the reply (reasoning tokens count in this limit).
    max_output_tokens=4096,
)
# Print the text of the reply.
print(response.output_text)

## Stream a response

Set `stream=True` to receive the text in small parts as the model writes it.

In [ ]:
# Send a request that returns the reply as a stream of events.
stream = client.responses.create(
    # Choose the deployed model that answers.
    model=DEPLOYMENT,
    # Send the system message and the user message.
    input=[
        # The system message asks for short answers.
        {"role": "system", "content": "You are a succinct assistant."},
        # The user message asks for a short list.
        {"role": "user",   "content": "List 3 prompt tuning levers with a one-liner each."}
    ],
    # Limit the length of the reply.
    max_output_tokens=4096,
    # Ask for the reply in parts instead of one block.
    stream=True,
)

# Read each event as it arrives.
for event in stream:
    # Act only on events that carry new text.
    if event.type == "response.output_text.delta":
        # Print the new text on the same line, without a line break.
        print(event.delta, end="")

## Get a JSON response

The system message asks for JSON only. The `text` setting makes the model return valid JSON. The code then parses the reply. If the parse fails, the cell prints the error.

If your model does not support the `text` setting, remove it and validate the JSON in your code.

In [ ]:
# Import the module that parses JSON text.
import json

# Build the messages: one rule for the format and one task.
messages_input = [
    # The system message defines the JSON keys the reply must contain.
    {"role": "system", "content": "Return ONLY valid JSON with keys: summary (string), bullets (array)."},
    # The user message is the task.
    {"role": "user",   "content": "Summarize what the Chat playground is used for and give 3 bullets."}
]
# Send the request.
response = client.responses.create(
    # Choose the deployed model that answers.
    model=DEPLOYMENT,
    # Send the messages built above.
    input=messages_input,
    # Force the model to return valid JSON.
    text={"format": {"type": "json_object"}},
    # Limit the length of the reply.
    max_output_tokens=4096,
)
# Keep the text of the reply.
text = response.output_text
# Print the raw JSON text.
print(text)

# Try to parse the text as JSON.
try:
    # Convert the JSON text into a Python dictionary.
    obj = json.loads(text)
    # Print the keys to show that the parse worked.
    print("\nParsed JSON keys:", list(obj.keys()))
# Catch any error from the parse.
except Exception as e:
    # Print the error so you can see what went wrong.
    print("\nJSON parse failed:", e)

## Parameter sweep: reasoning effort and output length

This cell runs the same prompt six times. It changes `reasoning effort` and `max_output_tokens` each time. Compare the length and detail of the answers.

> **Note:** A reasoning model can use its whole token budget on reasoning. The answer is then empty. Raise `max_output_tokens` if that happens.

In [ ]:
# Set the one prompt that every run uses.
prompt = "Explain embeddings vs. fine-tuning for short-form classification in 3 bullets."
# Loop over three levels of reasoning effort.
for effort in ["low", "medium", "high"]:
    # Loop over two output-length limits.
    for mot in [200, 400]:
        # Send the request with this effort and this limit.
        res = client.responses.create(
            # Choose the deployed model that answers.
            model=DEPLOYMENT,
            # Send the prompt as a single user message.
            input=[{"role": "user", "content": prompt}],
            # Set how much the model reasons before it answers.
            reasoning={"effort": effort},
            # Set the maximum number of tokens in the reply.
            max_output_tokens=mot,
        )
        # Print a header that shows the settings of this run.
        print(f"\n--- reasoning_effort={effort} max_output_tokens={mot}")
        # Print the answer, or a hint if the answer is empty.
        print(res.output_text or "(empty answer: the model used all tokens for reasoning. Raise max_output_tokens.)")

## Send a request with REST

This cell calls the `/responses` route with the `requests` library. It sends the same data as the SDK. The cell prints the HTTP status code and the first 2,000 characters of the JSON reply.

In [ ]:
# Import the HTTP library and the JSON module.
import requests, json

# Build the full address of the Responses route.
url = f"{ENDPOINT}/responses"
# Send the key in the header and tell the server the body is JSON.
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
# Build the request body.
payload = {
    # Name the deployed model.
    "model": DEPLOYMENT,
    # Send the system message and the user message.
    "input": [
        # The system message sets the role of the model.
        {"role": "system", "content": "You are a concise Azure assistant."},
        # The user message is the question.
        {"role": "user",   "content": "Name two cases to prefer a serverless API deployment."}
    ],
    # Limit the length of the reply.
    "max_output_tokens": 250
}
# Send the POST request and stop waiting after 60 seconds.
resp = requests.post(url, headers=headers, json=payload, timeout=60)
# Print the HTTP status code (200 means success).
print(resp.status_code)
# Print the first 2,000 characters of the reply, formatted for reading.
print(json.dumps(resp.json(), indent=2)[:2000])

## Troubleshooting

| Status code | Cause | Action |
|---|---|---|
| 401 | The key is wrong. | Copy the key from the **Details** tab again. |
| 404 | The endpoint or the deployment name is wrong. | Check `ENDPOINT` and `DEPLOYMENT`. |
| 429 | You sent too many requests. | Wait a few seconds. Run the cell again. |

## Conclusion

In this notebook, you did these tasks:

- Installed the OpenAI Python SDK.
- Sent a basic request and a streaming request to `gpt-5.4-mini`.
- Got a JSON response and parsed it.
- Ran a parameter sweep with different reasoning effort and output length values.
- Sent a request with REST.